# 04 · Street view on-road greenery per LSOA and ward
Street view images (about 58 m apart along roads) each carry the share of their pixels that are trees (`treePercent`, from a SegNet model).
- **On-road images** are those inside the road strips from 02 (same roads and strip width as the aerial measures).
- `GSV_on_road` = mean tree share of the on-road images in the area. With evenly spaced images this weights roads by their length, the counterpart of the aerial `on_road_green_share`.
- `GSV_on_road_weighted` = the same images, weighted by the road-centrality weight `choice_log` of the busiest road within the strip (`edge_choice` in `02_SAT_greenery_per_segment_workers.py`, also used by the aerial weighted measure).

Each image is assigned to the area it falls in, so it counts once.

**Inputs:** `data/raw_data/GSV/GLA_Tree.csv` (`cent_x`, `cent_y` in EPSG:27700, `treePercent`); `SAT_greenery_per_segment_tiles/road_buffers.gpkg` (from 02); `data/raw_data/M25/M25_seg_p.shp`.
**Output:** `data/greenery_metrics/GSV_greenery_per_{lsoa,ward}.csv`: `GSV_on_road`, `GSV_on_road_weighted`, `samples`.
**Runtime:** about 5 minutes.

In [ ]:
import importlib
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

sys.path.insert(0, str(Path.cwd().resolve()))
seg = importlib.import_module('02_SAT_greenery_per_segment_workers')
tot = importlib.import_module('01_SAT_greenery_total_workers')
GSV_PATH = seg.REPO_ROOT / 'data/raw_data/GSV/GLA_Tree.csv'
OUT_CSV = {label: seg.REPO_ROOT / f'data/greenery_metrics/GSV_greenery_per_{label}.csv' for label in ('lsoa', 'ward')}
CONFIGS = {'lsoa': tot.LSOA_CONFIG, 'ward': tot.WARD_CONFIG}

## 1. On-road images and their road-centrality weight

In [ ]:
gsv = pd.read_csv(GSV_PATH)
images = gpd.GeoDataFrame(gsv[['treePercent']], geometry=gpd.points_from_xy(gsv['cent_x'], gsv['cent_y']), crs=seg.CRS)
buffers = gpd.read_file(seg.BUFFERS_PATH, columns=[])
images = images[images.index.isin(gpd.sjoin(images, buffers, predicate='within').index)]
edges = seg.edge_choice()
edges['geometry'] = edges.buffer(seg.BUFFER_M)
busiest = gpd.sjoin(images, edges[['choice_log', 'geometry']], predicate='within').groupby(level=0)['choice_log'].max()
images['choice_log'] = busiest.reindex(images.index).fillna(0.0)
print(f'on-road images: {len(images):,} / {len(gsv):,} | with a weight > 0: {(images.choice_log > 0).sum():,}')

## 2. Images to areas → SAVE

In [ ]:
results = {}
for label, cfg in CONFIGS.items():
    area_id = cfg.area_id_col
    areas = tot.load_areas(cfg)[[area_id, 'geometry']].to_crs(seg.CRS)
    j = gpd.sjoin(images, areas, predicate='within')
    g = j.assign(tree_w=j['treePercent'] * j['choice_log']).groupby(area_id)
    scores = pd.DataFrame({
        'GSV_on_road': g['treePercent'].mean(),
        'GSV_on_road_weighted': g['tree_w'].sum() / g['choice_log'].sum().where(lambda s: s > 0),
        'samples': g.size(),
    })
    results[label] = areas[[area_id]].merge(scores, left_on=area_id, right_index=True, how='left')
    results[label].to_csv(OUT_CSV[label], index=False)
    print(f'[{label}] wrote {OUT_CSV[label].name} | areas: {len(areas):,}, with images: {results[label].samples.notna().sum():,}')

## Validation

In [ ]:
for label, df in results.items():
    assert df[CONFIGS[label].area_id_col].is_unique
    assert df[['GSV_on_road', 'GSV_on_road_weighted']].stack().between(0, 1).all()
    sat = pd.read_csv(seg.REPO_ROOT / f'data/greenery_metrics/SAT_greenery_on_off_road_{label}.csv')
    m = df.merge(sat, on=CONFIGS[label].area_id_col)
    print(f'[{label}] r(street view, aerial share) = {m.GSV_on_road.corr(m.on_road_green_share):.3f}, '
          f'weighted: {m.GSV_on_road_weighted.corr(m.on_road_green_share_weighted):.3f}')